# Week 2: Decomposition & Features
## In-Class Exercises

**Objective.** Split a series into trend, seasonality, and remainder; handle more than one seasonal period; and turn a decomposition into a handful of *features* that let you triage hundreds of series at once.

### How this notebook works

Three parts, each building on the one before it.

| Part | Format | Content |
| --- | --- | --- |
| 1 | Walkthrough | STL on the airline series, raw and logged, and what the remainder tells you. |
| 2 | Blanks we fill in together | MSTL on a series with daily *and* weekly seasonality. |
| 3 | On your own, ~12 min | Trend and seasonal strength across a panel, then rank it. |

In [ ]:
!pip install -q pandas numpy plotly matplotlib statsmodels scipy scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL, MSTL
from statsmodels.graphics.tsaplots import plot_acf

pio.templates.default = "plotly_white"   # try "plotly_dark", "ggplot2", "simple_white"

# Series and components are plotly. Correlograms stay with statsmodels' plot_acf.

URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
air = pd.read_csv(URL, parse_dates=["Month"], index_col="Month")["Passengers"].asfreq("MS")
air.head()

In [ ]:
def components_frame(res, label):
    """STL/MSTL result -> tidy frame with one row per (date, component)."""
    parts = {"observed": res.observed, "trend": res.trend, "resid": res.resid}
    if isinstance(res.seasonal, pd.DataFrame):
        parts.update({f"seasonal_{c}": res.seasonal[c] for c in res.seasonal})
    else:
        parts["seasonal"] = res.seasonal
    return (pd.DataFrame(parts)
              .rename_axis("date").reset_index()
              .melt(id_vars="date", var_name="component", value_name="value")
              .assign(fit=label))

---
## Part 1. STL, and why the transform comes first

STL splits $y_t$ into trend $T_t$, season $S_t$, and remainder $R_t$, additively:

$$y_t = T_t + S_t + R_t$$

*Additively* is doing a lot of work here. The airline series has seasonal swings that grow with the level, which additive STL cannot represent. It tries anyway, and the failure lands in the remainder.

Watch the remainder panel in both fits.

In [ ]:
stl_raw = STL(air, seasonal=13).fit()
stl_log = STL(np.log(air), seasonal=13).fit()

comp = pd.concat([components_frame(stl_raw, "raw"), components_frame(stl_log, "log")])

fig = px.line(comp, x="date", y="value", facet_row="component", facet_col="fit",
              height=800, title="STL components: raw vs. log")
fig.update_yaxes(matches=None)                       # each component keeps its own scale
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
plot_acf(stl_raw.resid, lags=36, ax=ax[0], title="Remainder ACF: raw")
plot_acf(stl_log.resid, lags=36, ax=ax[1], title="Remainder ACF: log")
plt.tight_layout()
plt.show()

print("remainder std, raw:", round(stl_raw.resid.std(), 3))
print("remainder std, log (as % of level):", round(100 * stl_log.resid.std(), 3))

**Notice:**

1. The raw remainder fans out, small early and large late. A fixed additive seasonal shape is too small in 1949 and too big in 1960.
2. The raw remainder still has ACF structure at the seasonal lags. Leftover seasonality means the decomposition did not finish its job.
3. On the log scale the seasonal amplitude is constant, additive STL fits, and the remainder looks close to noise.

**Rule of thumb:** if the seasonal swing scales with the level, decompose the log. On the log scale, an additive decomposition *is* a multiplicative one.

---
## Part 2. Two seasonal periods at once

Hourly business data usually has a daily cycle (period 24) inside a weekly cycle (period 168). One `seasonal` argument cannot express both; MSTL fits them in sequence.

We build a series with a known answer so the decomposition can be checked against ground truth.

In [ ]:
rng = np.random.default_rng(7)
n_hours = 24 * 7 * 12          # 12 weeks
idx = pd.date_range("2024-01-01", periods=n_hours, freq="h")
t = np.arange(n_hours)

true_trend = 50 + 0.004 * t
true_daily = 10 * np.sin(2 * np.pi * t / 24 - 1.5)
true_weekly = 6 * (np.isin(idx.dayofweek, [5, 6])).astype(float) * -1   # weekend dip
noise = rng.normal(scale=1.5, size=n_hours)

demand = pd.Series(true_trend + true_daily + true_weekly + noise, index=idx, name="demand")
px.line(demand.iloc[:24 * 14].rename_axis("date").reset_index(), x="date", y="demand",
        title="Simulated hourly demand, first two weeks").show()

In [ ]:
# TODO - fit MSTL with BOTH periods.
#   Hint: MSTL(demand, periods=(24, 168)).fit()
res = ...

# res.seasonal is a DataFrame with one column per period.
res.seasonal.head()

<details>
<summary><b>Show the line</b></summary>

```python
res = MSTL(demand, periods=(24, 168)).fit()
```
</details>

In [ ]:
three_weeks = components_frame(res, "MSTL").query("date < @demand.index[24 * 21]")

fig = px.line(three_weeks, x="date", y="value", facet_row="component", height=700,
              title="MSTL components, first three weeks")
fig.update_yaxes(matches=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

In [ ]:
# Now the diagnostic: re-fit with ONLY the daily period and see where the weekly cycle goes.
# TODO - fit STL/MSTL with periods=(24,) and plot the remainder's ACF out to 200 lags.
res_daily_only = ...

# YOUR PLOT HERE

<details>
<summary><b>Solution and what to look for</b></summary>

```python
res_daily_only = MSTL(demand, periods=(24,)).fit()

fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
plot_acf(res.resid, lags=200, ax=ax[0], title="Remainder ACF: periods (24, 168)")
plot_acf(res_daily_only.resid, lags=200, ax=ax[1], title="Remainder ACF: period (24,) only")
plt.tight_layout()
plt.show()
```

Dropping the weekly period does not delete the weekly cycle; it relocates it into the remainder, as a spike at lag 168 and its multiples. Structure in the remainder always means the same thing: *something you did not model is still in the data.*
</details>

---
## Part 3. Features from a decomposition

About 12 minutes.

With one series you plot it. With a thousand you compute a few numbers per series and sort. The two workhorses come straight out of STL:

$$F_T = \max\!\left(0,\ 1 - \frac{\operatorname{Var}(R_t)}{\operatorname{Var}(T_t + R_t)}\right)
\qquad
F_S = \max\!\left(0,\ 1 - \frac{\operatorname{Var}(R_t)}{\operatorname{Var}(S_t + R_t)}\right)$$

Both live in $[0, 1)$. Near 0 means the component explains almost nothing; near 1 means it explains almost everything.

**Tasks.**

1. Finish `features(y, period)` so it returns trend strength, seasonal strength, and the remainder's standard deviation.
2. Run it across the panel of 24 series generated in the next cell.
3. Sort by seasonal strength, then plot the top 2 and bottom 2 and confirm they look the way the numbers say.
4. One series is pure noise and one is pure trend. Identify both from the feature table alone, without plotting.

In [ ]:
def make_panel(n_series=24, n=120, seed=3):
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2015-01-01", periods=n, freq="MS")
    t = np.arange(n)
    out = {}
    for i in range(n_series):
        trend_amp = rng.choice([0.0, 0.05, 0.3, 1.0])
        seas_amp = rng.choice([0.0, 1.0, 5.0, 12.0])
        y = (100 + trend_amp * t
             + seas_amp * np.sin(2 * np.pi * t / 12 + rng.uniform(0, 6))
             + rng.normal(scale=3.0, size=n))
        out[f"s{i:02d}"] = y
    # two ringers, so you know the answer exists
    out["s00"] = 100 + rng.normal(scale=3.0, size=n)              # pure noise
    out["s01"] = 100 + 1.2 * t + rng.normal(scale=3.0, size=n)    # pure trend
    return pd.DataFrame(out, index=idx)

panel = make_panel()

first_four = (panel.iloc[:, :4].rename_axis("date").reset_index()
                   .melt(id_vars="date", var_name="series", value_name="value"))
fig = px.line(first_four, x="date", y="value", facet_row="series", height=600,
              title="First four panel series")
fig.update_yaxes(matches=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

In [ ]:
def features(y, period=12):
    """Return {'trend_strength', 'seasonal_strength', 'resid_sd'} for one series."""
    res = STL(pd.Series(y), period=period, seasonal=13).fit()
    var_r = np.var(res.resid)
    # TODO - fill in the two strength calculations using the formulas above
    trend_strength = ...
    seasonal_strength = ...
    return {"trend_strength": trend_strength,
            "seasonal_strength": seasonal_strength,
            "resid_sd": np.std(res.resid)}


# feat = pd.DataFrame({c: features(panel[c]) for c in panel}).T
# feat.round(3).sort_values("seasonal_strength", ascending=False)

In [ ]:
# YOUR CODE HERE - tasks 2, 3 and 4

<details>
<summary><b>Solution</b></summary>

```python
def features(y, period=12):
    res = STL(pd.Series(y), period=period, seasonal=13).fit()
    var_r = np.var(res.resid)
    trend_strength = max(0.0, 1 - var_r / np.var(res.trend + res.resid))
    seasonal_strength = max(0.0, 1 - var_r / np.var(res.seasonal + res.resid))
    return {"trend_strength": trend_strength,
            "seasonal_strength": seasonal_strength,
            "resid_sd": np.std(res.resid)}

feat = pd.DataFrame({c: features(panel[c].to_numpy()) for c in panel}).T
ranked = feat.sort_values("seasonal_strength", ascending=False)
print(ranked.round(3))

extremes = list(ranked.index[:2]) + list(ranked.index[-2:])
tidy = (panel[extremes].rename_axis("date").reset_index()
             .melt(id_vars="date", var_name="series", value_name="value"))
fig = px.line(tidy, x="date", y="value", facet_row="series", height=700,
              title="Top 2 and bottom 2 by seasonal strength")
fig.update_yaxes(matches=None)
fig.show()

# Task 4: pure noise = both strengths near 0; pure trend = high trend, low seasonal.
print("\npure noise candidate :", feat.eval("trend_strength + seasonal_strength").idxmin())
print("pure trend candidate:", feat.query("seasonal_strength < 0.2").trend_strength.idxmax())
```

**The point.** Two numbers per series turn an unreadable panel into a ranked list. This is how you decide which series get a hand-tuned model and which get seasonal naive, and how M4-style entries route series to model families.
</details>

---
## Wrap-up

1. **Transform, then decompose.** Additive machinery on multiplicative data pushes error into the remainder.
2. **Structure in the remainder is a to-do list.** A spike at lag 168 means you forgot the weekly cycle.
3. **Features turn a panel into a ranked list.** Two numbers per series decide where your modeling time goes.

Next week: ETS and ARIMA.